# Knowledge representation

How knowledge is stored decides what a program can conclude from it. Two classic answers organise concepts in hierarchies, so that what is true of a general concept is inherited by the specific ones.

## Semantic networks

A **semantic network** is a graph of concepts joined by labelled relations: *a penguin is a bird*, *a bird is an animal*. A property stated for a concept is **inherited** by everything below it, **unless a more specific concept says otherwise**: birds fly, but penguins don't. Reasoning that a later, more specific fact can overturn is called *default* or *non-monotonic* reasoning, and the rule here is simple: the closest answer up the `is_a` chain wins.

Each fact is a triple, subject, relation and value, the same shape as today's **knowledge graphs**, such as Wikidata.

In [1]:
animal_facts = [
    ("bird", "is_a", "animal"), ("mammal", "is_a", "animal"),
    ("penguin", "is_a", "bird"), ("sparrow", "is_a", "bird"), ("dog", "is_a", "mammal"),
    ("animal", "breathes", True),
    ("bird", "flies", True), ("bird", "covering", "feathers"),
    ("mammal", "covering", "fur"),
    ("penguin", "flies", False), ("penguin", "swims", True),
]


def look_up(facts, concept, relation):
    for subject, fact_relation, value in facts:
        if subject == concept and fact_relation == relation:
            return value
    parents = [value for subject, fact_relation, value in facts if subject == concept and fact_relation == "is_a"]
    for parent in parents:
        value = look_up(facts, parent, relation)
        if value is not None:
            return value
    return None

In [2]:
for concept in ["sparrow", "penguin", "dog"]:
    print(f"{concept:8} flies: {look_up(animal_facts, concept, 'flies')}, "
          f"covering: {look_up(animal_facts, concept, 'covering')}, "
          f"breathes: {look_up(animal_facts, concept, 'breathes')}")

sparrow  flies: True, covering: feathers, breathes: True
penguin  flies: False, covering: feathers, breathes: True
dog      flies: None, covering: fur, breathes: True


The sparrow inherits *flies* from bird, the penguin's own fact overrides it, and every animal inherits *breathes* from the top of the hierarchy. The dog doesn't fly, but the network can't say so: `None` means *unknown*, not *false*.

## Frames

A **frame** (Marvin Minsky, 1974) describes a stereotypical object or situation as a set of **slots** with values, and inherits the slots it doesn't fill from a more general frame. If that sounds like a class with attributes and a parent class, it is: frames are one of the ideas behind **object-oriented programming**.

In [3]:
vehicle = {"parent": None, "wheels": 4, "engine": True, "doors": 4}
car = {"parent": vehicle, "fuel": "petrol"}
sports_car = {"parent": car, "doors": 2, "top_speed_kmh": 250}
motorcycle = {"parent": vehicle, "wheels": 2, "doors": 0}


def slot(frame, name):
    while frame is not None:
        if name in frame:
            return frame[name]
        frame = frame["parent"]
    return None


for name, frame in [("car", car), ("sports car", sports_car), ("motorcycle", motorcycle)]:
    print(f"{name:11} wheels: {slot(frame, 'wheels')}, doors: {slot(frame, 'doors')}, engine: {slot(frame, 'engine')}")

car         wheels: 4, doors: 4, engine: True
sports car  wheels: 4, doors: 2, engine: True
motorcycle  wheels: 2, doors: 0, engine: True


Python classes do the same lookup, through their class attributes and the inheritance chain:

In [4]:
class Vehicle:
    wheels, engine, doors = 4, True, 4


class Car(Vehicle):
    fuel = "petrol"


class SportsCar(Car):
    doors, top_speed_kmh = 2, 250


class Motorcycle(Vehicle):
    wheels, doors = 2, 0


print("SportsCar doors:", SportsCar.doors, "| wheels:", SportsCar.wheels, "| fuel:", SportsCar.fuel)
print("Motorcycle doors:", Motorcycle.doors, "| engine:", Motorcycle.engine)

SportsCar doors: 2 | wheels: 4 | fuel: petrol
Motorcycle doors: 0 | engine: True


The object-oriented design of [tic-tac-toe's level 3](https://github.com/avidaldo/python-tic-tac-toe/tree/master/03_oop) uses the same mechanism: every kind of player inherits from one `Player` class and fills in its own way of choosing a move.